In [8]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

RANDOM_STATE = 111

In [9]:
import pandas as pd

X=pd.read_csv('/home/um40/LB2_project_Group_1/Feature_Selection/feature.csv')
# X=df.drop(columns='id',axis=1)

X.sort_values(by='id', inplace=True)

df=pd.read_csv('/home/um40/LB2_project_Group_1/Data_preparation/Cleaned_merged_dataset.tsv',sep='\t')

df = df.sort_values(by='ID')

y= df['class']
X= X.drop(columns='id', axis=1)
fold= df['fold']

feature_names=X.columns.to_list()




K_GRID=[len(feature_names)//4,len(feature_names)//3,len(feature_names)//2,len(feature_names)]
C_GRID=[1,4,10,16]


GRID=[]
for k in K_GRID:
    for c in C_GRID:
        GRID.append((k,c,np.round(0.5/k,4))) 
        GRID.append((k,c,np.round(1/k,4))) 
        GRID.append((k,c,np.round(2/k,4))) 
        GRID.append((k,c,np.round(4/k,4))) 

In [10]:
def train_and_test(X_train, y_train, X_test, y_test, new_model,c,gamma):
    # train a classifier on (X_train, y_train) and return its accuracy on (X_test, y_test)
    
    if X_train.shape[1] == 0:
        # no feature selected: the best we can do is predict the most frequent class
        majority = np.bincount(y_train).argmax()
        return np.mean(y_test == majority)
    # NB: this previous part is just for Lasso

    # 1. learn the scaling (mean, std of each column) on the TRAINING data only
    scaler = StandardScaler().fit(X_train) # --> zeta scores = subtract the mean and divide by std (dependency from data, we compute mean & std on training deta => NOT USE TEST DATA TO COMPUTE THESE METRICS)
    # 2. train the classifier on the scaled training data
    model = new_model(c,gamma)
    model.fit(scaler.transform(X_train), y_train)
    # 3. scale the test data with the SAME mean and std, predict, measure accuracy
    y_pred = model.predict(scaler.transform(X_test))
    return accuracy_score(y_test, y_pred)

In [11]:
def choose_setting(select, grid, X, y, new_model,test_fold,fold):
    # inner cross-validation on (X_train, y_train) only; returns the best setting
    # and the mean validation accuracy of every setting
    train_fold=[k for k in range(1,6) if k!=test_fold]
    mean_val_acc = {}
    for k,c,gamma in grid: # --> for each set of hyperparam values in the grid
        val_accs = []
        for j in train_fold:

            in_val=fold[fold==j].index.to_list()
            in_train=fold[(fold!=j) &( fold>0)].index.to_list()
    
            # selection sees the inner-training part only
            cols = select(X.iloc[in_train], y.iloc[in_train], k) #--> select the best features
            acc = train_and_test(X.iloc[in_train][:, cols], y.iloc[in_train],
                                 X.iloc[in_val][:, cols],   y.iloc[in_val], new_model,c,gamma) #--> train a new model with the subset of best features, then test it on validation fold
            val_accs.append(acc)
        mean_val_acc[k] = np.mean(val_accs)

    best = max(mean_val_acc, key=mean_val_acc.get)   # ties -> the first (smallest) one
    # if we have two combination of features with same accuracy, always choose the one with fewer features (= less model's params, less overfit)
    return best, mean_val_acc

In [12]:
results = []   # one entry per method

def nested_cv(name, family, select, grid, new_model,X,y,fold,verbose=True, store=True):
    
    start = time.time()
    test_accs, chosen, selected_sets, inner_curves = [], [], [], []
    # selected_sets --> selected set of feature for each fold (?)

    for i in range(1,6):

        test_idx=fold[fold==i].index.to_list()
        train_idx=fold[(fold!=i) &( fold>0)].index.to_list()


        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_test,  y_test  = X.iloc[test_idx],  y.iloc[test_idx]

        # STEP 1 - inner loop: choose the setting using the training part only
        if len(grid) == 1:
            best, curve = grid[0], {}          # nothing to choose --> test on user input (in this case the grid contains just one value)
        else:
            best, curve = choose_setting(select, grid, X, y, new_model,i,fold) #--> choose best setting (= #best_features to keep) on TRAIN SET

        # STEP 2 - redo the selection on the whole outer-training part
        cols = select(X_train, y_train, best) #--> given the #best_features (just a number!), select which are the #best most significant features on TRAIN TEST

        # STEP 3 - train on the outer-training part, test ONCE on the outer-test fold
        acc = train_and_test(X_train[:, cols], y_train, X_test[:, cols], y_test, new_model) #--> train a new model with the subset of best features, then test it on validation fold

        test_accs.append(acc)
        chosen.append(best)
        selected_sets.append(set(feature_names[cols]))
        inner_curves.append(curve)
        if verbose:
            print(f"  outer fold {fold}: chosen setting = {best:<5}  "
                  f"features kept = {len(cols):2d}   test accuracy = {acc:.3f}")

    seconds = time.time() - start
    test_accs = np.array(test_accs)
    if verbose:
        print(f"{name}: {test_accs.mean():.3f} ± {test_accs.std():.3f}   ({seconds:.1f}s)\n")

    entry = dict(method=name, family=family, chosen=chosen,
                 k=[len(s) for s in selected_sets], accuracy=test_accs.mean(),
                 std=test_accs.std(), seconds=seconds, fold_accs=test_accs,
                 sets=selected_sets, inner_curves=inner_curves, select=select, grid=grid)
    if store:
        results.append(entry)
    return entry

In [13]:
def select_rf(X_train, y_train, k):
    rf = RandomForestClassifier(n_estimators=100, random_state=0)
    rf.fit(X_train, y_train)
    ranking = np.argsort(rf.feature_importances_)[::-1]   # most important first (= gini importance)
    return ranking[:k]

def select_all(X_train, y_train, setting):
    return np.arange(X_train.shape[1])

def new_svm(c,gamma):
    return SVC(kernel="rbf", C=c, gamma=gamma)

In [14]:
rf = nested_cv("Embedded: RF importance", "embedded", select_rf, GRID,new_svm,X,y,fold)

TypeError: '(slice(None, None, None), array([ 2, 13,  0, 15,  1, 14, 16, 10,  9]))' is an invalid key

In [ ]:
baseline = nested_cv("Baseline (all features)", "-", select_all, grid=["all"])

In [ ]:
svm_all = nested_cv("RBF SVM, all features", "-", select_all, ["all"],
                    new_model=new_svm, store=False)
svm_rf  = nested_cv("RBF SVM, RF top-k", "embedded", select_rf, GRID,
                    new_model=new_svm, store=False)

In [ ]:
gain_lr  = rf["accuracy"] - baseline["accuracy"]
gain_svm = svm_rf["accuracy"] - svm_all["accuracy"]
print(f"Logistic regression: all {baseline['accuracy']:.3f} -> RF top-k {rf['accuracy']:.3f}   "
      f"gain {gain_lr:+.3f}")
print(f"RBF SVM            : all {svm_all['accuracy']:.3f} -> RF top-k {svm_rf['accuracy']:.3f}   "
      f"gain {gain_svm:+.3f}")